# Nuclei Segmentation U-Net Model Training

## 1. Mount to the google drive

In [ ]:
#import all necesarry libraries
from google.colab import drive
import os
import numpy as np
import cv2
import glob
import torch
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
drive.mount('/content/drive')

DATA_DIR = '/content/drive/My Drive/Dataset/data'
METADATA_DIR = os.path.join(DATA_DIR, "metadata")
MASK_DIR = os.path.join(DATA_DIR, "masks")

OUTPUT_MASK_DIR = os.path.join(DATA_DIR, "binary_masks")

SPLITS_TO_PROCESS = ["training.txt", "validation.txt", "test.txt"]

#os.listdir(DATA_DIR)

Mounted at /content/drive


## 2. Preprocess data, convert to binary

In [ ]:
def create_binary_mask(mask_path, output_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

    if mask is None:
        print(f"Warning: Could not read mask {mask_path}")
        return

    binary_mask = (mask > 0).astype(np.uint8)

    cv2.imwrite(output_path, binary_mask)

def process_all_masks():
    print(f"Starting mask preprocessing...")

    os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
    print(f"Saving binary masks to: {OUTPUT_MASK_DIR}")

    total_processed = 0

    for split_file in SPLITS_TO_PROCESS:
        split_filepath = os.path.join(METADATA_DIR, split_file)
        print(f"\nProcessing split: {split_file}...")

        if not os.path.exists(split_filepath):
            print(f"Warning: Split file not found: {split_filepath}")
            continue

        with open(split_filepath, 'r') as f:
            filenames = [line.strip() for line in f.readlines()]

        for basename in filenames:


            mask_filename = basename

            input_path = os.path.join(MASK_DIR, mask_filename)
            output_path = os.path.join(OUTPUT_MASK_DIR, mask_filename)

            if os.path.exists(input_path):
                create_binary_mask(input_path, output_path)
                total_processed += 1
            else:
                print(f"Warning: Mask file not found: {input_path}")

    print(f"\nPreprocessing complete. Processed {total_processed} masks.")

process_all_masks()

## 3. Dataset

In [ ]:
IMAGE_DIR = os.path.join(DATA_DIR, "images")
MASK_DIR = os.path.join(DATA_DIR, "binary_masks")
METADATA_DIR = os.path.join(DATA_DIR, "metadata")

IMG_WIDTH = 256
IMG_HEIGHT = 256

class NucleiDataset(Dataset):
    def __init__(self, split_file="training.txt"):
        print(f"Loading dataset from: {split_file}")
        self.image_dir = IMAGE_DIR
        self.mask_dir = MASK_DIR

        # Read the list of filenames
        split_filepath = os.path.join(METADATA_DIR, split_file)
        with open(split_filepath, 'r') as f:
            self.filenames = [line.strip().split('.')[0] for line in f.readlines()]

        print(f"Found {len(self.filenames)} images.")

    def __len__(self):
        #num of samples
        return len(self.filenames)

    def __getitem__(self, idx):
        base_name = self.filenames[idx]

        img_path = os.path.join(self.image_dir, f"{base_name}.tif")
        image = cv2.imread(img_path, cv2.IMREAD_UNCHANGED) # 16-bit


        mask_path = os.path.join(self.mask_dir, f"{base_name}.png")
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) #8bits grayscale

        image_resized = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT),
                                   interpolation=cv2.INTER_AREA)
        mask_resized = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT),
                                  interpolation=cv2.INTER_NEAREST)


        image_tensor = TF.to_tensor(image_resized.astype(np.float32) / 65535.0)


        mask_tensor = TF.to_tensor(mask_resized.astype(np.float32) / 1.0)

        return image_tensor, mask_tensor

if __name__ == "__main__":

    print("Testing the NucleiDataset...")

    train_dataset = NucleiDataset(split_file="training.txt")

    if len(train_dataset) > 0:
        print(f"\nSuccessfully loaded {len(train_dataset)} training samples.")

        image_tensor, mask_tensor = train_dataset[0]

        print(f"\n--- Sample 0 ---")
        print(f"Image Tensor Shape: {image_tensor.shape}")
        print(f"Image Tensor DataType: {image_tensor.dtype}")
        print(f"Image Tensor Min: {image_tensor.min()}, Max: {image_tensor.max()}")

        print(f"\nMask Tensor Shape: {mask_tensor.shape}")
        print(f"Mask Tensor DataType: {mask_tensor.dtype}")
        print(f"Mask Tensor Unique Values: {torch.unique(mask_tensor)}")


        assert image_tensor.shape == (1, IMG_HEIGHT, IMG_WIDTH)
        assert mask_tensor.shape == (1, IMG_HEIGHT, IMG_WIDTH)
        assert mask_tensor.max() <= 1.0
        assert mask_tensor.min() >= 0.0

        print("\nTest PASSED!")

    else:
        print("Test FAILED: Dataset is empty or could not be loaded.")

Testing the NucleiDataset...
Loading dataset from: training.txt
Found 100 images.

Successfully loaded 100 training samples.

--- Sample 0 ---
Image Tensor Shape: torch.Size([1, 256, 256])
Image Tensor DataType: torch.float32
Image Tensor Min: 0.0019684138242155313, Max: 0.027557794004678726

Mask Tensor Shape: torch.Size([1, 256, 256])
Mask Tensor DataType: torch.float32
Mask Tensor Unique Values: tensor([0.])

Test PASSED!


## 4. U-Net modeling

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

#u-net model
class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        #encoder
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)

        #decoder
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)

        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        #final
        logits = self.outc(x)
        return logits


print("Testing the U-Net model...")
model = UNet(n_channels=1, n_classes=1)

dummy_image = torch.randn(1, 1, 256, 256)

output = model(dummy_image)

print(f"\n--- Model Test ---")
print(f"Input shape: {dummy_image.shape}")
print(f"Output shape: {output.shape}")

assert output.shape == (1, 1, 256, 256)

print("\nTest PASSED.")

Testing the U-Net model...

--- Model Test ---
Input shape: torch.Size([1, 1, 256, 256])
Output shape: torch.Size([1, 1, 256, 256])

Test PASSED.


## 5. Model Training
Train the model with different number of epochs [10, 20, 50, 100, 150]

### 10 epochs

In [ ]:
NUM_EPOCHS = 10         #num of time loop over data
BATCH_SIZE = 4          #num of img process at once
LEARNING_RATE = 1e-4    # how fast the model learn
#MODEL_SAVE_PATH = "best_model_10.pth"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Assignment2/best_model_10.pth"

save_dir = os.path.dirname(MODEL_SAVE_PATH)

if save_dir:
    os.makedirs(save_dir, exist_ok=True)

#device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#dataset loading
train_dataset = NucleiDataset(split_file="training.txt")
val_dataset = NucleiDataset(split_file="validation.txt")

#dataset loader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False # No need to shuffle validation data
)


model = UNet(n_channels=1, n_classes=1).to(device)

#loss function
criterion = nn.BCEWithLogitsLoss()

#optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Starting training for {NUM_EPOCHS} epochs...")

best_val_loss = float('inf') # tracking best model

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()      # update model weights

        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss += loss.item() * images.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved to {MODEL_SAVE_PATH}")

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best model saved at {MODEL_SAVE_PATH}")

Using device: cuda
Loading dataset from: training.txt
Found 100 images.
Loading dataset from: validation.txt
Found 50 images.
Starting training for 10 epochs...
Epoch [1/10] | Train Loss: 0.5671 | Val Loss: 0.6198
New best model saved to /content/drive/MyDrive/Assignment2/best_model_10.pth
Epoch [2/10] | Train Loss: 0.4236 | Val Loss: 0.3579
New best model saved to /content/drive/MyDrive/Assignment2/best_model_10.pth
Epoch [3/10] | Train Loss: 0.3868 | Val Loss: 0.3964
Epoch [4/10] | Train Loss: 0.3647 | Val Loss: 0.3695
Epoch [5/10] | Train Loss: 0.3564 | Val Loss: 0.5497
Epoch [6/10] | Train Loss: 0.3455 | Val Loss: 0.3470
New best model saved to /content/drive/MyDrive/Assignment2/best_model_10.pth
Epoch [7/10] | Train Loss: 0.3170 | Val Loss: 0.3191
New best model saved to /content/drive/MyDrive/Assignment2/best_model_10.pth
Epoch [8/10] | Train Loss: 0.2999 | Val Loss: 0.3012
New best model saved to /content/drive/MyDrive/Assignment2/best_model_10.pth
Epoch [9/10] | Train Loss: 0.2

### 20 epochs

In [ ]:
NUM_EPOCHS = 20         #num of time loop over data
BATCH_SIZE = 4          #num of img process at once
LEARNING_RATE = 1e-4    # how fast the model learn
MODEL_SAVE_PATH = "/content/drive/MyDrive/Assignment2/best_model_20.pth"

save_dir = os.path.dirname(MODEL_SAVE_PATH)

if save_dir:
    os.makedirs(save_dir, exist_ok=True)

#device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#dataset loading
train_dataset = NucleiDataset(split_file="training.txt")
val_dataset = NucleiDataset(split_file="validation.txt")

#dataset loader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False # No need to shuffle validation data
)


model = UNet(n_channels=1, n_classes=1).to(device)

#loss function
criterion = nn.BCEWithLogitsLoss()

#optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Starting training for {NUM_EPOCHS} epochs...")

best_val_loss = float('inf') # tracking best model

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()      # update model weights

        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss += loss.item() * images.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved to {MODEL_SAVE_PATH}")

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best model saved at {MODEL_SAVE_PATH}")

Using device: cuda
Loading dataset from: training.txt
Found 100 images.
Loading dataset from: validation.txt
Found 50 images.
Starting training for 20 epochs...
Epoch [1/20] | Train Loss: 0.4965 | Val Loss: 0.6133
New best model saved to /content/drive/MyDrive/Assignment2/best_model_20.pth
Epoch [2/20] | Train Loss: 0.3732 | Val Loss: 0.3061
New best model saved to /content/drive/MyDrive/Assignment2/best_model_20.pth
Epoch [3/20] | Train Loss: 0.3457 | Val Loss: 0.3178
Epoch [4/20] | Train Loss: 0.3293 | Val Loss: 0.3287
Epoch [5/20] | Train Loss: 0.3064 | Val Loss: 0.3019
New best model saved to /content/drive/MyDrive/Assignment2/best_model_20.pth
Epoch [6/20] | Train Loss: 0.2891 | Val Loss: 0.2924
New best model saved to /content/drive/MyDrive/Assignment2/best_model_20.pth
Epoch [7/20] | Train Loss: 0.2738 | Val Loss: 0.2729
New best model saved to /content/drive/MyDrive/Assignment2/best_model_20.pth
Epoch [8/20] | Train Loss: 0.3056 | Val Loss: 0.2846
Epoch [9/20] | Train Loss: 0.2

### 50 epochs

In [ ]:
NUM_EPOCHS = 50         #num of time loop over data
BATCH_SIZE = 4          #num of img process at once
LEARNING_RATE = 1e-4    # how fast the model learn
MODEL_SAVE_PATH = "/content/drive/MyDrive/Assignment2/best_model_50.pth"

save_dir = os.path.dirname(MODEL_SAVE_PATH)

if save_dir:
    os.makedirs(save_dir, exist_ok=True)

#device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#dataset loading
train_dataset = NucleiDataset(split_file="training.txt")
val_dataset = NucleiDataset(split_file="validation.txt")

#dataset loader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False # No need to shuffle validation data
)


model = UNet(n_channels=1, n_classes=1).to(device)

#loss function
criterion = nn.BCEWithLogitsLoss()

#optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Starting training for {NUM_EPOCHS} epochs...")

best_val_loss = float('inf') # tracking best model

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()      # update model weights

        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss += loss.item() * images.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved to {MODEL_SAVE_PATH}")

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best model saved at {MODEL_SAVE_PATH}")

Using device: cpu
Loading dataset from: training.txt
Found 100 images.
Loading dataset from: validation.txt
Found 50 images.
Starting training for 50 epochs...
Epoch [1/50] | Train Loss: 0.5161 | Val Loss: 0.5067
New best model saved to /content/drive/MyDrive/Assignment2/best_model_50.pth
Epoch [2/50] | Train Loss: 0.3935 | Val Loss: 0.3224
New best model saved to /content/drive/MyDrive/Assignment2/best_model_50.pth
Epoch [3/50] | Train Loss: 0.3687 | Val Loss: 0.3940
Epoch [4/50] | Train Loss: 0.3385 | Val Loss: 0.3638
Epoch [5/50] | Train Loss: 0.3178 | Val Loss: 0.3187
New best model saved to /content/drive/MyDrive/Assignment2/best_model_50.pth
Epoch [6/50] | Train Loss: 0.3018 | Val Loss: 0.2998
New best model saved to /content/drive/MyDrive/Assignment2/best_model_50.pth
Epoch [7/50] | Train Loss: 0.2879 | Val Loss: 0.2865
New best model saved to /content/drive/MyDrive/Assignment2/best_model_50.pth
Epoch [8/50] | Train Loss: 0.2739 | Val Loss: 0.2651
New best model saved to /conten

### 100 epochs

In [ ]:
NUM_EPOCHS = 100         #num of time loop over data
BATCH_SIZE = 4          #num of img process at once
LEARNING_RATE = 1e-4    # how fast the model learn
MODEL_SAVE_PATH = "/content/drive/MyDrive/Assignment2/best_model_100.pth"

save_dir = os.path.dirname(MODEL_SAVE_PATH)

if save_dir:
    os.makedirs(save_dir, exist_ok=True)

#device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#dataset loading
train_dataset = NucleiDataset(split_file="training.txt")
val_dataset = NucleiDataset(split_file="validation.txt")

#dataset loader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False # No need to shuffle validation data
)


model = UNet(n_channels=1, n_classes=1).to(device)

#loss function
criterion = nn.BCEWithLogitsLoss()

#optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Starting training for {NUM_EPOCHS} epochs...")

best_val_loss = float('inf') # tracking best model

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()      # update model weights

        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss += loss.item() * images.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved to {MODEL_SAVE_PATH}")

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best model saved at {MODEL_SAVE_PATH}")

Using device: cuda
Loading dataset from: training.txt
Found 100 images.
Loading dataset from: validation.txt
Found 50 images.
Starting training for 100 epochs...
Epoch [1/100] | Train Loss: 0.4196 | Val Loss: 0.4518
New best model saved to /content/drive/MyDrive/Assignment2/best_model_100.pth
Epoch [2/100] | Train Loss: 0.2903 | Val Loss: 0.2659
New best model saved to /content/drive/MyDrive/Assignment2/best_model_100.pth
Epoch [3/100] | Train Loss: 0.2582 | Val Loss: 0.2597
New best model saved to /content/drive/MyDrive/Assignment2/best_model_100.pth
Epoch [4/100] | Train Loss: 0.2401 | Val Loss: 0.2309
New best model saved to /content/drive/MyDrive/Assignment2/best_model_100.pth
Epoch [5/100] | Train Loss: 0.2247 | Val Loss: 0.2239
New best model saved to /content/drive/MyDrive/Assignment2/best_model_100.pth
Epoch [6/100] | Train Loss: 0.2122 | Val Loss: 0.6136
Epoch [7/100] | Train Loss: 0.2036 | Val Loss: 0.1991
New best model saved to /content/drive/MyDrive/Assignment2/best_model_

### 150 epochs

In [ ]:
NUM_EPOCHS = 150         #num of time loop over data
BATCH_SIZE = 4          #num of img process at once
LEARNING_RATE = 1e-4    # how fast the model learn
MODEL_SAVE_PATH = "/content/drive/MyDrive/Assignment2/best_model_150.pth"

save_dir = os.path.dirname(MODEL_SAVE_PATH)

if save_dir:
    os.makedirs(save_dir, exist_ok=True)

#device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#dataset loading
train_dataset = NucleiDataset(split_file="training.txt")
val_dataset = NucleiDataset(split_file="validation.txt")

#dataset loader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False # No need to shuffle validation data
)


model = UNet(n_channels=1, n_classes=1).to(device)

#loss function
criterion = nn.BCEWithLogitsLoss()

#optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Starting training for {NUM_EPOCHS} epochs...")

best_val_loss = float('inf') # tracking best model

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()      # update model weights

        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss += loss.item() * images.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved to {MODEL_SAVE_PATH}")

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best model saved at {MODEL_SAVE_PATH}")

Using device: cuda
Loading dataset from: training.txt
Found 100 images.
Loading dataset from: validation.txt
Found 50 images.
Starting training for 150 epochs...
Epoch [1/150] | Train Loss: 0.3965 | Val Loss: 0.5122
New best model saved to /content/drive/MyDrive/Assignment2/best_model_150.pth
Epoch [2/150] | Train Loss: 0.2841 | Val Loss: 0.2179
New best model saved to /content/drive/MyDrive/Assignment2/best_model_150.pth
Epoch [3/150] | Train Loss: 0.2535 | Val Loss: 0.2309
Epoch [4/150] | Train Loss: 0.2400 | Val Loss: 0.2674
Epoch [5/150] | Train Loss: 0.2252 | Val Loss: 0.2223
Epoch [6/150] | Train Loss: 0.2049 | Val Loss: 0.2009
New best model saved to /content/drive/MyDrive/Assignment2/best_model_150.pth
Epoch [7/150] | Train Loss: 0.1915 | Val Loss: 0.1863
New best model saved to /content/drive/MyDrive/Assignment2/best_model_150.pth
Epoch [8/150] | Train Loss: 0.1806 | Val Loss: 0.1752
New best model saved to /content/drive/MyDrive/Assignment2/best_model_150.pth
Epoch [9/150] | 

## 6. Model Evaluation
evaluate the performance of each models trained with different number of epochs

In [ ]:
MODEL_BASE_DIR = "/content/drive/MyDrive/Assignment2/"
MODEL_FILES = [
    "best_model_10.pth",
    "best_model_20.pth",
    "best_model_50.pth",
    "best_model_100.pth",
    "best_model_150.pth"
]

BASE_OUTPUT_DIR = "/content/drive/MyDrive/Assignment2/predictions"

TEST_SPLIT_FILE = "test.txt"
BATCH_SIZE = 1

print(f"Loading evaluation script...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading test data from {TEST_SPLIT_FILE}...")
test_dataset = NucleiDataset(split_file=TEST_SPLIT_FILE)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
print(f"Found {len(test_dataset)} test images.")

def dice_score(pred, target, smooth=1e-6):
    """Calculates the Dice score for a single batch."""
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return dice


results = {}

for model_filename in MODEL_FILES:
    print(f"\n--- Evaluating Model: {model_filename} ---")

    MODEL_PATH = os.path.join(MODEL_BASE_DIR, model_filename)


    model_name_without_ext = model_filename.split('.')[0]
    OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, model_name_without_ext)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Saving predictions to: {OUTPUT_DIR}")


    model = UNet(n_channels=1, n_classes=1).to(device)

    if not os.path.exists(MODEL_PATH):
        print(f"Warning: Model file not found, skipping: {MODEL_PATH}")
        continue

    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    model.eval()

    total_dice = 0.0
    with torch.no_grad():
        for i, (image, mask) in enumerate(tqdm(test_loader, desc="Testing")):
            image = image.to(device)
            mask = mask.to(device)

            output = model(image)
            pred_prob = torch.sigmoid(output)
            pred_mask = (pred_prob > 0.5).float()

            total_dice += dice_score(pred_mask, mask).item()

            basename = test_dataset.filenames[i]
            save_path = os.path.join(OUTPUT_DIR, f"{basename}_pred.png")

            pred_mask_np = pred_mask.cpu().squeeze().numpy()
            pred_mask_img = (pred_mask_np * 255).astype(np.uint8)
            cv2.imwrite(save_path, pred_mask_img)

    avg_dice = total_dice / len(test_loader)
    print(f"Result for {model_filename}: Dice Score = {avg_dice:.4f}")
    results[model_filename] = avg_dice

print("\n--- Evaluation Complete: All Models ---")
print("------------------------------------------")
print(f"{'Model Name':<25} | {'Dice Score':<10}")
print("------------------------------------------")
for model_name, score in results.items():
    print(f"{model_name:<25} | {score:<10.4f}")
print("------------------------------------------")

Loading evaluation script...
Using device: cuda
Loading test data from test.txt...
Loading dataset from: test.txt
Found 50 images.
Found 50 test images.

--- Evaluating Model: best_model_10.pth ---
Saving predictions to: /content/drive/MyDrive/Assignment2/predictions/best_model_10


Testing: 100%|██████████| 50/50 [00:02<00:00, 22.85it/s]


Result for best_model_10.pth: Dice Score = 0.7200

--- Evaluating Model: best_model_20.pth ---
Saving predictions to: /content/drive/MyDrive/Assignment2/predictions/best_model_20


Testing: 100%|██████████| 50/50 [00:02<00:00, 22.56it/s]


Result for best_model_20.pth: Dice Score = 0.8600

--- Evaluating Model: best_model_50.pth ---
Saving predictions to: /content/drive/MyDrive/Assignment2/predictions/best_model_50


...

Result for best_model_50.pth: Dice Score = 1.0000

--- Evaluating Model: best_model_100.pth ---
Saving predictions to: /content/drive/MyDrive/Assignment2/predictions/best_model_100


Testing: 100%|██████████| 50/50 [00:01<00:00, 25.08it/s]


Result for best_model_100.pth: Dice Score = 1.0000

--- Evaluating Model: best_model_150.pth ---
Saving predictions to: /content/drive/MyDrive/Assignment2/predictions/best_model_150


Testing: 100%|██████████| 50/50 [00:01<00:00, 25.36it/s]

Result for best_model_150.pth: Dice Score = 1.0000

--- Evaluation Complete: All Models ---
------------------------------------------
Model Name                | Dice Score
------------------------------------------
best_model_10.pth         | 0.7200    
best_model_20.pth         | 0.8600    
best_model_50.pth         | 1.0000    
best_model_100.pth        | 1.0000    
best_model_150.pth        | 1.0000    
------------------------------------------
